<a href="https://colab.research.google.com/github/InduwaraGayashan001/LangChain/blob/main/Prompt_Templates_%26_Chains.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup


In [1]:
from google.colab import userdata

In [ ]:
!pip install langchain-community langchain-openai

#Prompt Templates

In [7]:
from langchain_core.prompts import PromptTemplate

# Prompt Tempalate
prompt = PromptTemplate(
    input_variables=['topic'],
    template="Generate 5 interesting facts about {topic}"
)

print(prompt.format(topic="Sri Lanka"))

Generate 5 interesting facts about Sri Lanka


# Using Chains with Prompt Templates

## Simple Chain

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Configure the LLM
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)

# Prompt Tempalate
prompt = PromptTemplate(
    input_variables=['topic'],
    template="Generate 5 interesting facts about {topic}"
)

chain = prompt | llm | StrOutputParser()
result = chain.invoke({'topic':'Sri Lanka'})
print(result)

Certainly! Here are five interesting facts about Sri Lanka:

1. **Rich Biodiversity**: Sri Lanka is recognized as one of the world's biodiversity hotspots, home to an array of wildlife, including elephants, leopards, and various endemic species. The country has several national parks, such as Yala and Udawalawe, which are popular for wildlife safaris.

2. **Ceylon Tea**: Sri Lanka is famous for its tea, known as Ceylon tea, which is considered some of the best in the world. The island’s diverse climate and altitude contribute to the unique flavors of its tea. The tea industry is a significant part of the country’s economy and culture.

3. **Ancient Civilization**: Sri Lanka has a rich history, with civilizations that date back over 2,500 years. The ancient city of Anuradhapura, a UNESCO World Heritage Site, features well-preserved ruins, including the vast stupas and the sacred Bodhi tree believed to be a sapling of the tree under which Buddha attained enlightenment.

4. **Cultural Her

## Sequential Chain

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Configure the LLM
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)


# Step 1: Prompt to get the largest country in a continent
prompt1 = PromptTemplate(
    input_variables=['continent'],
    template="What is the largest country in {continent}? Just give me the name of the country."
)

# Step 2: Prompt to get the capital of the country from step 1
prompt2 = PromptTemplate(
    input_variables=['country'],
    template="What is the capital of {country}? Just give me the name of the city."
)

# Output parser to get clean string responses
parser = StrOutputParser()

# Build the sequential chain
sequential_chain = prompt1 | llm | parser | prompt2 | llm | parser

# Invoke the chain with an input
result = sequential_chain.invoke({"continent": "Asia"})
print(result)

Moscow.


## Parallel Chain

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

# String output parser
parser = StrOutputParser()

# Configure the first LLM (OpenAI via GitHub Marketplace)
llm1 = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)

# Configure the second LLM (OpenAI via GitHub Marketplace)
llm2 = ChatOpenAI(
    model="openai/gpt-5-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)

# Prompt 1: Historical facts
prompt1 = PromptTemplate(
    input_variables=['topic'],
    template="Give 3 interesting facts about {topic} from a historical perspective."
)

# Prompt 2: Cultural facts
prompt2 = PromptTemplate(
    input_variables=['topic'],
    template="Give 3 interesting facts about {topic} from a cultural perspective."
)

# Prompt 3: Combine outputs
combine_prompt = PromptTemplate(
    input_variables=['historical', 'cultural'],
    template="Here are two sets of facts:\nHistorical: {historical}\nCultural: {cultural}\n\nWrite a concise summary combining both historical and cultural facts."
)

# Create a parallel chain for prompts 1 and 2
parallel_chain = RunnableParallel({
    'historical': prompt1 | llm1 | parser,
    'cultural': prompt2 | llm1 | parser
})

# Combine outputs using the third prompt and LLM2
final_chain = parallel_chain | combine_prompt | llm2 | parser

# Invoke the final chain
result = final_chain.invoke({"topic": "Sri Lanka"})
print(result)


Sri Lanka’s history spans over 2,500 years, with advanced ancient kingdoms like Anuradhapura and Polonnaruwa known for sophisticated irrigation, monumental architecture, and the early adoption of Buddhism that shaped society and art. Its strategic position on historic maritime trade routes made it a crossroads of Greek, Roman, Arab and Asian contacts, while three centuries of European colonial rule (Portuguese, Dutch, British) and plantation crops such as tea transformed its economy and landscape until independence in 1948. Today the island’s population reflects a diverse mix of ethnic and religious communities—chiefly Sinhalese and Tamils, with Buddhists, Hindus, Muslims and Christians—expressed through traditional arts (notably Kandyan dance and batik) and vibrant festivals like the Sinhala–Tamil New Year and the Kandy Esala Perahera. Together, these historical layers and cultural practices create Sri Lanka’s distinctive and enduring cultural tapestry.


## Conditional Chain

In [18]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

# Configure the LLM (OpenAI via GitHub Marketplace)
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=userdata.get('GITHUB_TOKEN'),
    base_url="https://models.github.ai/inference"
)

# String output parser
parser = StrOutputParser()

# Define a Pydantic model for structured sentiment output
class Feedback(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(
        description="Sentiment of the feedback"
    )

# Parser to enforce structured output
parser2 = PydanticOutputParser(pydantic_object=Feedback)

# Prompt to classify sentiment
prompt1 = PromptTemplate(
    template=(
        "Classify the sentiment of the following feedback as positive or negative.\n"
        "Feedback: {feedback}\n"
        "{format_instruction}"
    ),
    input_variables=['feedback'],
    partial_variables={
        'format_instruction': parser2.get_format_instructions()
    }
)

# Chain to classify sentiment
classifier_chain = prompt1 | llm | parser2

# Prompt for positive feedback response
prompt2 = PromptTemplate(
    template="Write an appropriate response to this positive feedback:\n{feedback}",
    input_variables=['feedback']
)

# Prompt for negative feedback response
prompt3 = PromptTemplate(
    template="Write an appropriate response to this negative feedback:\n{feedback}",
    input_variables=['feedback']
)

# Conditional branching logic
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt2 | llm | parser),
    (lambda x: x.sentiment == 'negative', prompt3 | llm | parser),
    RunnableLambda(lambda x: "Could not determine sentiment.")
)

# Full conditional chain
chain = classifier_chain | branch_chain

# Invoke the chain
print(chain.invoke({'feedback': 'This is a beautiful phone'}))
print(chain.invoke({'feedback': 'This is a terrible phone'}))


Thank you so much for your kind words! We’re thrilled to hear that you had a positive experience. Your feedback motivates us to keep striving for excellence. If there’s anything else you’d like to share or any suggestions you have, we’d love to hear them!
Thank you for your feedback. I’m truly sorry to hear that your experience didn’t meet your expectations. Your insights are important, and I’d like to learn more about what went wrong so we can improve. Please feel free to share additional details, and I’ll do my best to address your concerns. Thank you for your input, and I hope we can resolve this together.
